# Lekcija 18 (nastavak): Povezivanje potpisa odobravatelja s točnom akcijom

Ovaj bilježnik provjerava dvije potpisane izjave: učvršćeni ključ odobravatelja potpisao je točnu akciju, a učvršćeni ključ agenta potpisao je zahtjev koji sadrži istu akciju. Ne poziva davatelja usluge, ne troši odobrenje, niti dokazuje izvršenje. Generirani ključevi služe kao zamjena za zasebno registrirane ključeve odobravatelja i agenta; primjer ne izvodi WebAuthn niti dokazuje da je osoba bila prisutna.

Oba artefakta koriste ravnu omotnicu lekcije: tipizirani sadržaj potpisan s Ed25519 preko njegovih kanoničnih JCS bajtova, s strukturiranim objektom potpisa izvan tih potpisanih bajtova. Tip odobrenja, human.approval.v1, je edukativna kompozicija, a ne tip definiran u draft-farley-acta-signed-receipts.

Odvojeni registri učvršćenih ključeva drže ta dva tijela odvojena. Ugrađeni ključ ne može uspostaviti povjerenje. Samo za čitanje verificator također provjerava vezu akcije, isporučenu verziju politike i isteka u isporučenom vremenu provjere. Pravi izvršitelj mora zasebno implementirati trajnu, atomsku zaštitu od ponovnog izvođenja i uskladiti ishode uspjeha, neuspjeha i nepoznatog davatelja usluge.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.b64decode(s + "=" * ((4 - len(s) % 4) % 4), altchars=b"-_", validate=True)

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Točna radnja

Jedinica odobrenja je kanonski objekt radnje, a ne nejasna oznaka poput "odobri povrat." Njegov puni objekt i digest omogućuju provjeritelju da provjeri koju je radnju ključ odobravatelja potpisao. Ne utvrđuju što je osoba vidjela ili razumjela.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Jedna omotnica, dvije vlasti

Svaki potvrda je ravni tipizirani sadržaj plus objekt potpisa koji je izuzet od potpisanih bajtova. Zajednički verify_envelope pomoćnik koristi zapisnik pinanih ključeva pozivatelja:

- **Potvrda odobrenja**: ključ odobravatelja potpisuje cijelu radnju, njezin sažetak, verziju politike i vremenske oznake izdavanja/isteka.
- **Potvrda agenta**: ključ agenta potpisuje isti sažetak radnje, tvrdnju o ishodu i vremenu, te parent_approval_ref, hash odobrenja.

Zajednički action_digest je spoj. key_id je samo pokazivač za pretraživanje u pinanom zapisniku. Verifikacija ne troši niti jedan artefakt.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    key_id = sig_obj.get("key_id")
    signature = sig_obj.get("sig")
    if not isinstance(key_id, str) or not isinstance(signature, str):
        return (False, "signature malformed (key_id and sig must be strings)")
    try:
        pub = trusted_keys.get(key_id)
    except (TypeError, ValueError):
        return (False, "invalid pinned key registry")
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(signature))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (demo key)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## Verifikacija lanca potvrda samo za čitanje

verify_chain provjerava oba potpisa, kanonsku vezu akcije, svježinu politike/ključa i isticanje. Uspoređuje tvrdnje potvrde s danom akcijom i vremenom verifikacije. Primjer koristi normalizirane UTC vremenske oznake u istom ISO-8601 obliku.

Ponavljana verifikacija mora dati isti odgovor za iste ulaze i konfiguraciju povjerenja. To nije operacija slanja i ne pruža jednokratno izvršenje. Izvršitelj u produkciji treba vlastiti trajni status rezervacije/potrošnje na granici davatelja.


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, claimed_at, sk=agent_sk, key_id="agent-key-1"):
    claimed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": claimed_action,
        "action_digest": sha256_canonical(claimed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "claimed_performed",
        "claimed_at": claimed_at,
    }
    return sign_receipt(payload, sk, key_id)


def verify_chain(claimed_action, approval, agent_rcpt, now: str):
    """Read-only verification of signed claims, not execution or consumption.
    The example supplies normalized ISO-8601 UTC timestamps for comparison."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the claimed action
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the claimed action changed".
    claimed_digest = sha256_canonical(claimed_action)
    if approval.get("action_digest") != claimed_digest or approval.get("action") != claimed_action:
        return (False, "digest substitution: the approval binds a different canonical action than the claimed action")
    if agent_rcpt.get("action_digest") != claimed_digest or agent_rcpt.get("action") != claimed_action:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the claimed action")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before verification")

    return (True, "matching signed claims; execution and replay status not established")

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(verify_chain(action, approval, receipt, now="2026-07-08T15:04:07Z"))


(True, 'matching signed claims; execution and replay status not established')


## Što povezivanje hvata

Donje provjere odbacuju manipulacije, zamjene, krivotvorene potpise, neispravne potvrde, zastarjele politike/ključeve i istekle odobrenja. Također pokazuju da se valjani lanac može višekratno potvrditi. Višekratna provjera dokaza nije zaštita od ponovnog reproduciranja: izvršitelj mora zasebno odbiti dupliciranu dispatch naredbu.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the claimed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented as evidence for action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. Repeated offline verification is valid. It does not authorize a second dispatch.
print("repeat-verification ->", verify_chain(action, approval, receipt, NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (demo key)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved after approval. Verification uses the supplied current policy.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but verification came too late.
expired = human_approval(action, "alice@ops (demo key)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     claims action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (demo key)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # claiming `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the claimed action')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the claimed action')
repeat-verification -> (True, 'matching signed claims; execution and replay status not established')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the claimed action')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-rece

## Što ovo uspostavlja, a što ne uspostavlja

**Uspostavlja:** pod danim pričvršćenim ključevima i politikom, ključ odobravatelja je potpisao ovu točnu kanoničku radnju i ključ agenta potpisao je odgovarajuću izjavu vezanu uz to odobrenje. Provjera isteka koristi dano vrijeme verifikacije. Različiti razlozi za neuspjeh identificiraju zastarjelu ovlast ili promijenjenu tvrdnju.

**Ne uspostavlja:** da je čovjek vidio ili odobrio išta, da su potpisni ključevi bili sigurni, da su tvrdnje o vremenskim oznakama pouzdane ili da je pružatelj izvršio radnju. Ishod je tvrdnja potpisana od strane agenta, a ne potvrda pružatelja.

Ovaj bilježnik ne pokreće radnje niti ne koristi odobrenja. Zaštita od reprodukcije u produkciji zahtijeva trajan atomski prijelaz stanja na granici izvršitelja. Izgubljeni odgovor nakon što je pružatelj možda prihvatio radnju mora ostati neodređen dok se ne pomiri, a ne postati dozvola za ponovni pokušaj.

Dvije vrste potvrda namjerno dijele jedan put verifikacije omotnice pod zasebnim pričvršćenim registrima. To čini povezivanje neovisno provjerljivim bez pretvaranja verifikacije potvrde u ovlast za izvršenje.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
